# ST-Reason · 09 Statistics and figures

Reads `outputs/metrics/answers_all.csv` (written by notebook 08 — **run 08 first**) and produces the paper's tests, figures and LaTeX tables. **CPU only**, about a minute.

**How every comparison is tested.** Two runs are compared on **exactly the same questions** (those both answered), so each question gives a pair of outcomes.
- **Difference** in accuracy, with a 95% **cluster-bootstrap** confidence interval (videos are resampled, since questions about one video are not independent).
- **Exact McNemar test** on the discordant pairs (questions one run got right and the other wrong) for the p-value.
- **Holm correction** over the three pre-registered primary tests (H1, H2, H3). Every other comparison is reported as exploratory, without correction.
- **Minimum detectable difference** (80% power, α = 0.05) from the observed share of discordant pairs, so a non-significant result can be read as "no effect larger than X".

Every test is run on strict scoring (an unreadable reply is wrong) and on lenient scoring (a letter recovered from the reply text).

| Hypothesis | Comparison |
| --- | --- |
| **H1** (primary) memory structure | M4 vs M1, NExT-QA, strategy D; also under strategy A and on STAR |
| **H2** (primary) temporal gap | pause vs sham (10 s), M4, strategy D; also 25 s and M1 |
| **H3** (primary) reasoning strategy | D vs A, M4, NExT-QA; plus D and A vs A-SC on dev |
| **H4** perception vs reasoning | oracle memory vs CV memory (STAR, M4 and M1) |
| **H5** QLoRA | tuned vs untuned Qwen3-4B; tuned 4B vs untuned 8B |
| Baselines | M4 + D vs blind LLM, captions, VideoAgent-style memory, frames-only VLM |

In [ ]:
# --- Parameters ----------------------------------------------------------------------------------
N_BOOT = 5000
ALPHA = 0.05

In [ ]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '09_stats_figures'
ST_QUIET = True
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 200)
from scipy.stats import binomtest, norm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
MANIFEST_PATH = P['runs'] / 'manifest_09.csv'
DRIVE_LOG_PATH = LOG_PATH
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
MET = P['metrics'] if P['metrics'].exists() else P['pred']
FIG = P['figures'] if P['figures'].exists() else MET          # files only, no new folders on the shared Drive
TAB = P['tables'] if P['tables'].exists() else MET
WORK = P['local'] / 'stats'
WORK.mkdir(parents=True, exist_ok=True)
src = MET / 'answers_all.csv'
assert src.exists(), 'answers_all.csv not found in outputs/metrics: run notebook 08 first'
A = pd.read_csv(src, dtype={'video': str})
A['correct'] = A.correct.astype(bool); A['correct_lenient'] = A.correct_lenient.astype(bool)
RUNS = set(A.run)
RUN_LOG = Run(params={'step': 'stats', 'n_boot': N_BOOT, 'answers': len(A), 'source_mtime': src.stat().st_mtime})
print(f'{len(A)} answers, {len(RUNS)} runs, from {src}')
def save_out(df, name, folder):
    tmp = WORK / name
    df.to_csv(tmp, index=False) if name.endswith('.csv') else tmp.write_text(df)
    shutil.copyfile(tmp, folder / name)

## 1. The paired test

In [ ]:
RNG = np.random.default_rng(CFG['seed'])

def paired(run_a, run_b, key_a='base_uid', key_b='base_uid', score='correct'):
    """Compare run_a with run_b on the questions both answered. key_* chooses how questions are matched
    (base_uid matches a pause/sham version to its original question)."""
    if run_a not in RUNS or run_b not in RUNS:
        return None
    a = A[A.run == run_a].drop_duplicates(key_a).set_index(key_a)
    b = A[A.run == run_b].drop_duplicates(key_b).set_index(key_b)
    common = a.index.intersection(b.index)
    if len(common) < 5:
        return None
    ya, yb = a.loc[common, score].to_numpy(bool), b.loc[common, score].to_numpy(bool)
    vid = a.loc[common, 'video'].astype(str).to_numpy()
    d = ya.astype(float) - yb.astype(float)
    # cluster bootstrap over videos for the CI of the difference
    uv, inv = np.unique(vid, return_inverse=True)
    s = np.bincount(inv, weights=d, minlength=len(uv)); n = np.bincount(inv, minlength=len(uv)).astype(float)
    idx = RNG.integers(0, len(uv), size=(N_BOOT, len(uv)))
    boot = s[idx].sum(1) / n[idx].sum(1)
    b10 = int((ya & ~yb).sum()); b01 = int((~ya & yb).sum())
    p = binomtest(b10, b10 + b01, 0.5).pvalue if b10 + b01 > 0 else 1.0
    disc = (b10 + b01) / len(common)
    mde = (norm.ppf(1 - ALPHA / 2) + norm.ppf(0.8)) * np.sqrt(disc / len(common))   # McNemar normal approximation
    return {'a': run_a, 'b': run_b, 'score': score, 'n': len(common), 'videos': len(uv), 'acc_a': ya.mean(),
            'acc_b': yb.mean(), 'diff': d.mean(), 'ci_low': np.percentile(boot, 2.5), 'ci_high': np.percentile(boot, 97.5),
            'a_only': b10, 'b_only': b01, 'p': p, 'mde80': mde}

## 2. All comparisons

`a − b` is the accuracy of the first run minus the second, on their common questions. Primary tests carry a Holm-adjusted p-value.

In [ ]:
TESTS = [
    # (hypothesis, label, run a, run b, primary?, key_a, key_b)
    ('H1', 'M4 vs M1, NExT-QA, strategy D', 'main.M4.D', 'main.M1.D', True, 'base_uid', 'base_uid'),
    ('H1', 'M4 vs M1, NExT-QA, strategy A', 'main.M4.A', 'main.M1.A', False, 'base_uid', 'base_uid'),
    ('H1', 'M4 vs M1, STAR, strategy D', 'star.M4.D', 'star.M1.D', False, 'base_uid', 'base_uid'),
    ('H1', 'M4 vs M1, STAR, strategy A', 'star.M4.A', 'star.M1.A', False, 'base_uid', 'base_uid'),
    ('H2', 'sham vs pause 10 s, M4, D', 'gap_sham10.M4.D', 'gap_pause10.M4.D', True, 'base_uid', 'base_uid'),
    ('H2', 'sham vs pause 25 s, M4, D', 'gap_sham25.M4.D', 'gap_pause25.M4.D', False, 'base_uid', 'base_uid'),
    ('H2', 'sham vs pause 10 s, M1, D', 'gap_sham10.M1.D', 'gap_pause10.M1.D', False, 'base_uid', 'base_uid'),
    ('H2', 'sham vs pause 25 s, M1, D', 'gap_sham25.M1.D', 'gap_pause25.M1.D', False, 'base_uid', 'base_uid'),
    ('H2', 'original vs pause 10 s, M4, D', 'star.M4.D', 'gap_pause10.M4.D', False, 'base_uid', 'base_uid'),
    ('H3', 'D vs A, M4, NExT-QA', 'main.M4.D', 'main.M4.A', True, 'base_uid', 'base_uid'),
    ('H3', 'D vs A, M4, STAR', 'star.M4.D', 'star.M4.A', False, 'base_uid', 'base_uid'),
    ('H3', 'D vs A-SC, M4, dev', 'dev.M4.D', 'dev.M4.A-SC', False, 'base_uid', 'base_uid'),
    ('H3', 'A-SC vs A, M4, dev', 'dev.M4.A-SC', 'dev.M4.A', False, 'base_uid', 'base_uid'),
    ('H4', 'oracle vs CV memory, M4, STAR, D', 'star.ORACLE_M4.D', 'star.M4.D', False, 'base_uid', 'base_uid'),
    ('H4', 'oracle vs CV memory, M1, STAR, D', 'star.ORACLE_M1.D', 'star.M1.D', False, 'base_uid', 'base_uid'),
    ('H4', 'oracle M4 vs oracle M1, STAR, D', 'star.ORACLE_M4.D', 'star.ORACLE_M1.D', False, 'base_uid', 'base_uid'),
    ('H5', 'QLoRA vs untuned Qwen3-4B, NExT-QA, A', 'main.M4.A.q4b-lora', 'main.M4.A.q4b', False, 'base_uid', 'base_uid'),
    ('H5', 'QLoRA vs untuned Qwen3-4B, STAR, A', 'star.M4.A.q4b-lora', 'star.M4.A.q4b', False, 'base_uid', 'base_uid'),
    ('H5', 'QLoRA 4B vs untuned 8B, NExT-QA, A', 'main.M4.A.q4b-lora', 'main.M4.A', False, 'base_uid', 'base_uid'),
    ('H5', 'QLoRA 4B vs untuned 8B, STAR, A', 'star.M4.A.q4b-lora', 'star.M4.A', False, 'base_uid', 'base_uid'),
    ('Base', 'M4+D vs blind LLM, NExT-QA', 'main.M4.D', 'main.BLIND.A', False, 'base_uid', 'base_uid'),
    ('Base', 'M4+D vs blind LLM, STAR', 'star.M4.D', 'star.BLIND.A', False, 'base_uid', 'base_uid'),
    ('Base', 'M4+A vs captions+A, NExT-QA', 'main.M4.A', 'main.CAP.A', False, 'base_uid', 'base_uid'),
    ('Base', 'M4+D vs captions+A, STAR', 'star.M4.D', 'star.CAP.A', False, 'base_uid', 'base_uid'),
    ('Base', 'M4+A vs VideoAgent-style+A, NExT-QA', 'main.M4.A', 'main.VA.A', False, 'base_uid', 'base_uid'),
    ('Base', 'M4+D vs VideoAgent-style+A, STAR', 'star.M4.D', 'star.VA.A', False, 'base_uid', 'base_uid'),
    ('Base', 'M4+D vs frames-only VLM, NExT-QA', 'main.M4.D', 'main.VLM.F16', False, 'base_uid', 'base_uid'),
    ('Base', 'M4+D vs frames-only VLM, STAR', 'star.M4.D', 'star.VLM.F16', False, 'base_uid', 'base_uid'),
    ('Base', 'oracle M4+D vs frames-only VLM, STAR', 'star.ORACLE_M4.D', 'star.VLM.F16', False, 'base_uid', 'base_uid'),
]
rows = []
for hyp, label, a, b, prim, ka, kb in TESTS:
    for score in ['correct', 'correct_lenient']:
        r = paired(a, b, ka, kb, score)
        if r:
            rows.append({'hypothesis': hyp, 'test': label, 'primary': prim, **r})
        elif score == 'correct':
            print(f'skipped (a run is missing or has <5 common questions): {label}')
S = pd.DataFrame(rows)

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * p[i])); adj[i] = run
    return adj
S['p_holm'] = np.nan
for sc in ['correct', 'correct_lenient']:
    m = S.primary & (S.score == sc)
    if m.any():
        S.loc[m, 'p_holm'] = holm(S.loc[m, 'p'])
S['significant'] = np.where(S.primary, S.p_holm < ALPHA, S.p < ALPHA)
save_out(S.round(4), 'tests.csv', MET)

def fmt(r):
    star = '*' if r.significant else ''
    ph = f" (Holm {r.p_holm:.3f})" if r.primary else ''
    return (f"{r.acc_a:.3f} vs {r.acc_b:.3f} | diff {r.diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] | "
            f"n={r.n} | p={r.p:.4f}{ph}{star} | MDE {r.mde80:.3f}")
for sc, title in [('correct', 'STRICT scoring'), ('correct_lenient', 'LENIENT scoring')]:
    print(f'\n=== {title} (* = significant; primary tests use Holm) ===')
    for h, g in S[S.score == sc].groupby('hypothesis', sort=False):
        print(f'\n{h}')
        for r in g.itertuples():
            print(f"  {'[primary] ' if r.primary else ''}{r.test}: {fmt(r)}")

## 3. Plain-language verdicts

One line per hypothesis, from the strict primary test (and the lenient one where it changes the reading). A difference is called "no clear effect" when its CI includes 0; the minimum detectable difference says how large an effect the data could have found.

In [ ]:
def verdict(label, sc='correct'):
    r = S[(S.test == label) & (S.score == sc)]
    if not len(r):
        return f'{label}: not available'
    r = r.iloc[0]
    sig = r.significant
    dd = abs(r['diff'])
    word = ('higher' if r['diff'] > 0 else 'lower') if sig else 'no clear difference'
    return (f"{label}: {r.a} {word}{'' if not sig else f' by {dd:.3f}'} "
            f"(diff {r['diff']:+.3f}, 95% CI [{r.ci_low:+.3f}, {r.ci_high:+.3f}], n={r.n}"
            f"{', could detect ≥ %.3f' % r.mde80 if not sig else ''})")
LINES = []
for label in ['M4 vs M1, NExT-QA, strategy D', 'M4 vs M1, NExT-QA, strategy A', 'M4 vs M1, STAR, strategy A',
              'sham vs pause 10 s, M4, D', 'D vs A, M4, NExT-QA', 'oracle vs CV memory, M4, STAR, D',
              'QLoRA vs untuned Qwen3-4B, NExT-QA, A', 'QLoRA 4B vs untuned 8B, NExT-QA, A',
              'M4+D vs blind LLM, NExT-QA', 'M4+D vs frames-only VLM, NExT-QA']:
    for sc in ['correct', 'correct_lenient']:
        LINES.append(('strict   ' if sc == 'correct' else 'lenient  ') + verdict(label, sc))
print('\n'.join(LINES))
save_out('\n'.join(LINES) + '\n', 'verdicts.txt', MET)

## 4. Figures

Saved as PDF (for LaTeX) and PNG in `outputs/figures/`. Blue marks the systems built in this project, gray the baselines; NExT-QA and STAR use the same two colours in every figure.

In [ ]:
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#52514e',
                     'axes.labelcolor': '#0b0b0b', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
                     'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
                     'pdf.fonttype': 42, 'figure.dpi': 150})
C_NEXT, C_STAR, C_OURS, C_BASE, INK = '#2a78d6', '#eb6834', '#2a78d6', '#a3a29e', '#52514e'
QR = pd.read_csv(MET / 'qa_runs.csv') if (MET / 'qa_runs.csv').exists() else None
def acc_ci(run):
    if QR is None or run not in set(QR.run):
        return None
    r = QR[QR.run == run].iloc[0]
    return r.accuracy, r.ci_low, r.ci_high
def savefig(fig, name):
    for ext in ['pdf', 'png']:
        tmp = WORK / f'{name}.{ext}'
        fig.savefig(tmp, bbox_inches='tight')
        shutil.copyfile(tmp, FIG / f'{name}.{ext}')
    plt.close(fig)
    print('figure saved:', name)

# Fig. 1: main results, both datasets
SYSTEMS = [('Blind LLM', 'BLIND.A', False), ('Captions + A', 'CAP.A', False), ('VideoAgent-style + A', 'VA.A', False),
           ('M1 + D', 'M1.D', True), ('M4 + A', 'M4.A', True), ('M4 + D', 'M4.D', True),
           ('Frames-only VLM', 'VLM.F16', False)]
fig, ax = plt.subplots(figsize=(6.2, 3.2))
y = np.arange(len(SYSTEMS))
for off, ds, col, name in [(-0.18, 'main', C_NEXT, 'NExT-QA'), (0.18, 'star', C_STAR, 'STAR')]:
    for i, (lab, suf, ours) in enumerate(SYSTEMS):
        v = acc_ci(f'{ds}.{suf}')
        if v:
            ax.barh(i + off, v[0], height=0.34, color=col, alpha=1.0 if ours else 0.45, label=name if i == 5 else None)
            ax.errorbar(v[0], i + off, xerr=[[v[0] - v[1]], [v[2] - v[0]]], fmt='none', ecolor=INK, elinewidth=0.8, capsize=2)
ax.axvline(0.2, color=INK, lw=0.6, ls=':'); ax.text(0.205, len(SYSTEMS) - 0.45, 'chance (NExT-QA)', fontsize=7, color=INK)
ax.set_yticks(y); ax.set_yticklabels([s[0] for s in SYSTEMS]); ax.invert_yaxis()
ax.set_xlim(0, 1); ax.set_xlabel('Accuracy (95% cluster-bootstrap CI)')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
savefig(fig, 'fig1_main_results')

# Fig. 2: paired differences (forest plot), strict and lenient
F = S[S.hypothesis.isin(['H1', 'H3', 'H4', 'H5'])]
labs = list(dict.fromkeys(F.test))
fig, ax = plt.subplots(figsize=(6.2, 0.32 * len(labs) + 0.8))
for j, (sc, col, mk, dy) in enumerate([('correct', C_NEXT, 'o', -0.15), ('correct_lenient', C_STAR, 's', 0.15)]):
    g = F[F.score == sc].set_index('test')
    for i, l in enumerate(labs):
        if l in g.index:
            r = g.loc[l]
            ax.errorbar(r['diff'], i + dy, xerr=[[r['diff'] - r.ci_low], [r.ci_high - r['diff']]], fmt=mk, color=col,
                        ms=4, elinewidth=1, capsize=2, label=('strict' if sc == 'correct' else 'lenient') if i == 0 else None)
ax.axvline(0, color=INK, lw=0.8)
ax.set_yticks(range(len(labs))); ax.set_yticklabels(labs, fontsize=7.5); ax.invert_yaxis()
ax.set_xlabel('Accuracy difference, first minus second (95% CI)')
ax.legend(loc='lower right', frameon=False, fontsize=8)
savefig(fig, 'fig2_paired_differences')

# Fig. 3: perception gap (oracle vs CV memory on STAR) with the VLM for reference
P3 = [('CV memory', 'star.M1.D', 'star.M4.D'), ('Oracle memory', 'star.ORACLE_M1.D', 'star.ORACLE_M4.D')]
fig, ax = plt.subplots(figsize=(4.2, 2.8))
for i, (lab, m1, m4) in enumerate(P3):
    for off, run, col, name in [(-0.18, m1, '#a3a29e', 'M1'), (0.18, m4, C_OURS, 'M4')]:
        v = acc_ci(run)
        if v:
            ax.bar(i + off, v[0], width=0.34, color=col, label=name if i == 0 else None)
            ax.errorbar(i + off, v[0], yerr=[[v[0] - v[1]], [v[2] - v[0]]], fmt='none', ecolor=INK, elinewidth=0.8, capsize=2)
            ax.text(i + off, v[2] + 0.02, f'{v[0]:.2f}', ha='center', fontsize=7.5, color='#0b0b0b')
vl = acc_ci('star.VLM.F16')
if vl:
    ax.axhline(vl[0], color=C_STAR, lw=1, ls='--'); ax.text(-0.45, vl[0] + 0.015, 'frames-only VLM', fontsize=7, color=INK, ha='left')
ax.set_xticks(range(len(P3))); ax.set_xticklabels([p[0] for p in P3]); ax.set_ylim(0, 1.05); ax.set_ylabel('STAR accuracy')
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2)
savefig(fig, 'fig3_perception_gap')

# Fig. 4: QLoRA
P4 = [('Qwen3-4B', '.M4.A.q4b'), ('Qwen3-4B + QLoRA', '.M4.A.q4b-lora'), ('Qwen3-8B', '.M4.A')]
fig, ax = plt.subplots(figsize=(4.6, 2.6))
for i, (lab, suf) in enumerate(P4):
    for off, ds, col, name in [(-0.18, 'main', C_NEXT, 'NExT-QA'), (0.18, 'star', C_STAR, 'STAR')]:
        v = acc_ci(ds + suf)
        if v:
            ax.bar(i + off, v[0], width=0.34, color=col, label=name if i == 0 else None)
            ax.errorbar(i + off, v[0], yerr=[[v[0] - v[1]], [v[2] - v[0]]], fmt='none', ecolor=INK, elinewidth=0.8, capsize=2)
ax.set_xticks(range(len(P4))); ax.set_xticklabels([p[0] for p in P4]); ax.set_ylim(0, 0.8); ax.set_ylabel('Accuracy (strategy A, M4)')
ax.legend(frameon=False, loc='upper left', ncol=2)
savefig(fig, 'fig4_qlora')

# Fig. 5: accuracy by NExT-QA question type
TT = pd.read_csv(MET / 'qa_by_type.csv') if (MET / 'qa_by_type.csv').exists() else None
if TT is not None:
    runs5 = [('Blind LLM', 'main.BLIND.A', '#a3a29e'), ('M1 + D', 'main.M1.D', '#6250d6'), ('M4 + D', 'main.M4.D', C_NEXT),
             ('Frames-only VLM', 'main.VLM.F16', C_STAR)]
    groups = [k for k in ['causal', 'temporal', 'descriptive'] if k in set(TT[TT.run == 'main.M4.D'].group)]
    fig, ax = plt.subplots(figsize=(5.4, 2.6))
    w = 0.8 / len(runs5)
    for j, (lab, run, col) in enumerate(runs5):
        g = TT[TT.run == run].set_index('group')
        xs = [i - 0.4 + w * (j + 0.5) for i in range(len(groups))]
        ax.bar(xs, [g.accuracy.get(k, np.nan) for k in groups], width=w * 0.92, color=col, label=lab)
    ns = TT[TT.run == 'main.M4.D'].set_index('group').n
    ax.set_xticks(range(len(groups))); ax.set_xticklabels([f'{k}\n(n={int(ns.get(k, 0))})' for k in groups])
    ax.set_ylim(0, 1); ax.set_ylabel('NExT-QA accuracy')
    ax.legend(frameon=False, fontsize=7.5, ncol=4, loc='lower center', bbox_to_anchor=(0.5, 1.0))
    savefig(fig, 'fig5_by_question_type')

# Fig. 6: cost vs accuracy (one dot per run, NExT-QA)
if QR is not None:
    q = QR[(QR.qset == 'main') & QR.s_per_q.notna()]
    fig, ax = plt.subplots(figsize=(4.6, 3.0))
    ax.scatter(q.s_per_q, q.accuracy, s=28, color=C_NEXT, edgecolor='white', linewidth=1.5, zorder=3)
    for r in q.itertuples():
        ax.annotate(r.run.replace('main.', ''), (r.s_per_q, r.accuracy), fontsize=6.5, color=INK,
                    xytext=(4, 2), textcoords='offset points')
    ax.set_xlabel('Seconds per question (T4 GPU, LLM or VLM stage only)'); ax.set_ylabel('NExT-QA accuracy')
    ax.set_ylim(0, 1)
    savefig(fig, 'fig6_cost_accuracy')

## 5. LaTeX tables

`tab_main.tex` (main results) and `tab_tests.tex` (all paired tests, strict scoring) in `outputs/tables/`, ready for `\input{}` in the paper.

In [ ]:
def tex_escape(s):
    return str(s).replace('_', r'\_').replace('&', r'\&').replace('%', r'\%')
PT = pd.read_csv(MET / 'paper_main_table.csv') if (MET / 'paper_main_table.csv').exists() else None
if PT is not None:
    lines = [r'\begin{table}[t]', r'\centering', r'\caption{Accuracy with 95\% cluster-bootstrap CIs (shards 0--1).}',
             r'\label{tab:main}', r'\begin{tabular}{lcc}', r'\toprule', r'System & NExT-QA & STAR \\', r'\midrule']
    for r in PT.itertuples():
        cell = lambda x: tex_escape(x).replace(' (n=', r' {\scriptsize (n=').replace(')', ')}') if x != '—' else '--'
        lines.append(f'{tex_escape(r.system)} & {cell(r._2)} & {cell(r.STAR)} \\\\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    save_out('\n'.join(lines) + '\n', 'tab_main.tex', TAB)
st = S[S.score == 'correct']
lines = [r'\begin{table*}[t]', r'\centering', r'\caption{Paired comparisons on common questions (strict scoring). '
         r'Diff: accuracy of the first minus the second; CI: 95\% cluster bootstrap; p: exact McNemar; '
         r'primary tests Holm-adjusted.}', r'\label{tab:tests}', r'\begin{tabular}{llrrrll}', r'\toprule',
         r'Hyp. & Comparison & n & Diff & 95\% CI & p & MDE \\', r'\midrule']
for r in st.itertuples():
    p = f'{r.p_holm:.3f}$^\\dagger$' if r.primary else f'{r.p:.3f}'
    lines.append(f"{r.hypothesis} & {tex_escape(r.test)} & {r.n} & {r.diff:+.3f} & [{r.ci_low:+.3f}, {r.ci_high:+.3f}] & {p} & {r.mde80:.3f} \\\\")
lines += [r'\bottomrule', r'\multicolumn{7}{l}{\scriptsize $^\dagger$ Holm-adjusted over the three primary tests.}',
          r'\end{tabular}', r'\end{table*}']
save_out('\n'.join(lines) + '\n', 'tab_tests.tex', TAB)
print('tables saved: tab_main.tex, tab_tests.tex')
RUN_LOG.finish('done', notes=f'{len(S)} tests')
print('\nfigures:', sorted(p.name for p in FIG.glob('fig*.pdf')))

## What to send back

The printed tests (section 2), the verdicts (section 3), and a look at the figures in `outputs/figures/` (the PNG files open directly in Drive).